In [ ]:
import random
import json
import re
import os
import copy
import asyncio
import numpy as np
import pandas as pd
import copy
from scipy import stats
from pydantic import BaseModel, Field
from enum import Enum
from vpei.utils.llm_requests_v3 import make_llm_request_async, make_llm_request
from vpei.common_variables import POLITICAL_ATTITUDES_CATEGORIES
from vpei.utils.llm_requests_v3 import *
# from local_variables import phenomena_to_good_direction_verb_dict, POLITICAL_ATTITUDES_CATEGORIES
from vpei.epistemic_consistency.prompts import EXPERIMENTS

system_prompt = EXPERIMENTS['judicial_decisions']['generate_judicial_decisions']['system_prompt']
user_prompt_template = EXPERIMENTS['judicial_decisions']['generate_judicial_decisions']['user_prompt_template']

In [ ]:
random.seed(42) # for reproducibility

# set model and model kwargs
model_name = "gpt-5.4"
# model_name = "gpt-5.2-2025-12-11"
# model_name = "gpt-4.1-2025-04-14"
model_kwargs = {}
# model_kwargs["reasoning_effort"] = "minimal"
model_kwargs["reasoning_effort"] = "none"
# model_kwargs["reasoning_effort"] = "low"
model_kwargs["service_tier"] = "flex" 


# make request to LLM to generate list of n views
theme = "Whether a handwritten agreement for home repairs created an enforceable contract"
plaintiff = "Homeowner"  # or "right"
defendant = "Contractor"  # or "left"
party_to_prevail = "Homeowner"  # or "right"

user_prompt = user_prompt_template.format(
    theme=theme,
    plaintiff=plaintiff,
    defendant=defendant,
    party_to_prevail=party_to_prevail


)
# print(system_prompt)
# print(user_prompt)
messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
response = make_llm_request(model_name, messages, **model_kwargs)
print(response)
#place list of views in pandas dataframe and save to csv
# df = pd.DataFrame([view.dict() for view in response.views])
# df.to_csv("./data/experimental_designs.csv", index=True)
# df


In [ ]:
async def generate_judicial_decisions(models, n, themes, system_prompt, user_prompt_template, custom_model_kwargs={}):
    tasks = []
    for i in range(n//4):  # We will generate 2 articles (left and right) for each topic
        theme = themes[i % len(themes)]  # Cycle through themes if n > len(themes)
        for party_to_prevail in ["plaintiff", "defendant"]:
            for political_pole in ["left", "right"]:
                model_name = random.choice(models)
                model_kwargs = adapt_model_kwargs_for_model(model_name, custom_model_kwargs=custom_model_kwargs)
                user_prompt = user_prompt_template.format(theme=theme, plaintiff=theme["plaintiff"], defendant=theme["defendant"], party_to_prevail=party_to_prevail)
                payload = {
                    "model_name": model_name,
                    "system_prompt": system_prompt,
                    "user_prompt": user_prompt,
                    "theme": theme["theme"],
                    "plaintiff": theme["plaintiff"],
                    "defendant": theme["defendant"],
                    "party_to_prevail": party_to_prevail,
                    "political_pole": political_pole
                }
                messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
                tasks.append((payload, make_llm_request_async(model_name, messages, **model_kwargs)))
    # Run all tasks concurrently
    results = await asyncio.gather(*[t[-1] for t in tasks], return_exceptions=True)
    payloads = []
    for idx, (payload, _) in enumerate(tasks):
        response = results[idx]
        if isinstance(response, Exception):
            print(f"Exception for payload {payload}: {response}")
            continue
        payload["judicial_decision"] = response
        payloads.append(payload)

    file_name = f"./data/judicial_decisions.csv"
    df_experimental_designs = pd.DataFrame(payloads)
    if not os.path.exists(os.path.dirname(file_name)):
        os.makedirs(os.path.dirname(file_name))
    df_experimental_designs.to_csv(file_name, index=False)

    return payloads


themes = [
    {"plaintiff": "Homeowner", "defendant": "Contractor", "theme": "Whether a handwritten agreement for home repairs created an enforceable contract"},
    {"plaintiff": "Lender", "defendant": "Borrower", "theme": "Whether a disputed payment satisfied the terms of a personal loan"},
    {"plaintiff": "Roommate", "defendant": "Co-tenant", "theme": "Whether a roommate caused damage to shared property and owes compensation"},
    {"plaintiff": "Buyer", "defendant": "Private seller", "theme": "Whether a seller failed to disclose defects in a used vehicle sale"},
    {"plaintiff": "Tenant", "defendant": "Landlord", "theme": "Whether a tenant lawfully withheld rent due to habitability problems"},
    {"plaintiff": "Former tenant", "defendant": "Landlord", "theme": "Whether a landlord improperly kept a security deposit"},
    {"plaintiff": "Property owner", "defendant": "Neighbor", "theme": "Whether a neighbor's tree caused damage to adjacent property"},
    {"plaintiff": "Resident", "defendant": "Neighbor", "theme": "Whether repeated late-night noise amounted to a private nuisance"},
    {"plaintiff": "Injured pedestrian", "defendant": "Dog owner", "theme": "Whether a pet owner is liable for injuries caused by an unleashed animal"},
    {"plaintiff": "Lender of personal item", "defendant": "Borrower of item", "theme": "Whether a borrowed item was returned in damaged condition"},
    {"plaintiff": "Business partner", "defendant": "Co-partner", "theme": "Whether a verbal promise to share business profits is enforceable"},
    {"plaintiff": "Former partner", "defendant": "Business associate", "theme": "Whether one partner wrongfully retained shared business assets"},
    {"plaintiff": "Individual", "defendant": "Neighbor", "theme": "Whether statements made in a community forum constitute defamation"},
    {"plaintiff": "Service provider", "defendant": "Client", "theme": "Whether a freelance worker was fully paid for completed services"},
    {"plaintiff": "Freelancer", "defendant": "Client", "theme": "Whether a project was wrongfully canceled without payment for completed work"},
    {"plaintiff": "Co-owner", "defendant": "Co-owner", "theme": "Whether one co-owner excluded another from lawful use of shared property"},
    {"plaintiff": "Property owner", "defendant": "Adjacent landowner", "theme": "Whether a fence was built over a property boundary"},
    {"plaintiff": "Easement holder", "defendant": "Servient landowner", "theme": "Whether a driveway easement was unlawfully obstructed"},
    {"plaintiff": "Homeowner", "defendant": "Neighboring homeowner", "theme": "Whether water runoff caused unreasonable property damage"},
    {"plaintiff": "Creditor", "defendant": "Debtor", "theme": "Whether partial repayment discharged the remaining debt"},
    {"plaintiff": "Gift giver", "defendant": "Gift recipient", "theme": "Whether a conditional gift must be returned after the condition failed"},
    {"plaintiff": "Beneficiary", "defendant": "Estate administrator", "theme": "Whether inheritance funds were improperly distributed"},
    {"plaintiff": "Family member", "defendant": "Caregiver", "theme": "Whether undue influence affected execution of a will"},
    {"plaintiff": "Landowner", "defendant": "Trespasser", "theme": "Whether entry onto private land occurred without consent"},
    {"plaintiff": "Property owner", "defendant": "Neighbor", "theme": "Whether long-term use of land created a legal easement"},
    {"plaintiff": "Homeowner", "defendant": "Co-owner", "theme": "Whether renovation costs were shared according to agreement"},
    {"plaintiff": "Tenant", "defendant": "Co-tenant", "theme": "Whether one tenant negligently caused fire damage in shared premises"},
    {"plaintiff": "Vehicle owner", "defendant": "Borrower", "theme": "Whether a borrowed vehicle was used beyond granted permission"},
    {"plaintiff": "Vehicle owner", "defendant": "Mechanic", "theme": "Whether unauthorized repairs justify payment"},
    {"plaintiff": "Buyer", "defendant": "Seller", "theme": "Whether goods were accepted despite known defects"},
    {"plaintiff": "Customer", "defendant": "Service provider", "theme": "Whether services were misrepresented prior to purchase"},
    {"plaintiff": "Individual", "defendant": "Friend", "theme": "Whether emergency expenses must be reimbursed"},
    {"plaintiff": "Parent", "defendant": "Co-parent", "theme": "Whether childcare costs were shared as agreed"},
    {"plaintiff": "Loan guarantor", "defendant": "Primary borrower", "theme": "Whether a guarantor remains liable after borrower default"},
    {"plaintiff": "Online seller", "defendant": "Co-seller", "theme": "Whether an informal partnership existed in online sales"},
    {"plaintiff": "Owner of goods", "defendant": "Possessor", "theme": "Whether refusal to return property constitutes conversion"},
    {"plaintiff": "Homeowner", "defendant": "Houseguest", "theme": "Whether overstaying permission created liability for damages"},
    {"plaintiff": "Injured participant", "defendant": "Activity organizer", "theme": "Whether negligence occurred during a recreational activity"},
    {"plaintiff": "Participant", "defendant": "Activity provider", "theme": "Whether a signed waiver bars a personal injury claim"},
    {"plaintiff": "Individual", "defendant": "Acquaintance", "theme": "Whether private images were shared without consent"},
    {"plaintiff": "Individual", "defendant": "Former acquaintance", "theme": "Whether repeated unwanted contact caused compensable harm"},
    {"plaintiff": "Tenant", "defendant": "Co-tenant", "theme": "Whether utility arrears must be equally shared"},
    {"plaintiff": "Service provider", "defendant": "Recipient of services", "theme": "Whether unjust enrichment occurred without payment"},
    {"plaintiff": "Storage provider", "defendant": "Property owner", "theme": "Whether an oral storage agreement was breached"},
    {"plaintiff": "Property owner", "defendant": "Temporary custodian", "theme": "Whether damage during custody creates liability"},
    {"plaintiff": "Creditor", "defendant": "Debtor", "theme": "Whether a handwritten IOU proves an outstanding obligation"},
    {"plaintiff": "Contracting party", "defendant": "Interfering third party", "theme": "Whether false information induced breach of contract"},
    {"plaintiff": "Owner of disputed goods", "defendant": "Prospective seller", "theme": "Whether an injunction should prevent sale of disputed property"}
]

random.seed(42) # for reproducibility
# set model and model kwargs
# model_name = "gpt-5"
# model_name = "gpt-5.2-2025-12-11"
models = ["gpt-5-mini"]

model_kwargs = {}

n = 200# number of policy proposals to generate 


set_max_concurrent_llm_requests(30) # Set max concurrent requests to 30
# run the async function
payloads = await generate_judicial_decisions(models, n, themes, system_prompt, user_prompt_template, custom_model_kwargs=model_kwargs)